# B5 - caption/VQA LoRA adapter, thin run

Trains a QLoRA adapter on `Qwen/Qwen2-VL-2B-Instruct` (pinned revision) using the B1 slice, then scores the
untuned base model and the adapter on the same held-out BigEarthNet.txt `bench` sample.

**Before running**
1. Upload `satquery_b1_slice_v0.zip` (from `python ml/pack_for_kaggle.py`) as a **private** Kaggle Dataset and add it to this notebook.
2. Settings: Accelerator = GPU T4 x2 (only one card is used), Internet = **On** (pip and the model download need it).
3. Read `ml/b5_train_lora.py`'s header: the T4 has no bfloat16, so this runs in float16, which we have **not** tested. Watch `skipped_nonfinite` in the training log.

This is a recipe check, not a finished adapter. `MAX_STEPS` below is small on purpose.

In [ ]:
!pip install -q "transformers==5.17.0" "peft==0.21.0" "bitsandbytes==0.50.2" "accelerate==1.15.0"
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
print("gpu:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))

In [ ]:
import glob, os, pathlib
hits = glob.glob("/kaggle/input/**/ml/b5_train_lora.py", recursive=True)
assert hits, "dataset not found: add satquery_b1_slice_v0 as an input to this notebook"
PKG = pathlib.Path(hits[0]).parent.parent          # folder holding ml/ and data/
DATA = PKG / "data"
print("package:", PKG)
print({p.name: sum(1 for _ in open(p, encoding="utf-8")) for p in DATA.glob("*.jsonl")})

In [ ]:
# Dry run first: 3 optimiser steps. If this fails or reports skipped_nonfinite > 0, stop here.
os.environ["HF_HOME"] = "/kaggle/working/hf"
!python {PKG}/ml/b5_train_lora.py --data {DATA} --out /kaggle/working/dry --size 448 --max-steps 3 --grad-accum 2 --eval-every 0 --save-every 0

In [ ]:
# The thin training run. 300 steps x 16 examples is about 4,800 of the 17,955 available (~0.27 epoch).
MAX_STEPS = 300
!python {PKG}/ml/b5_train_lora.py --data {DATA} --out /kaggle/working/b5_run1 --size 448 --max-steps {MAX_STEPS} --grad-accum 16 --eval-every 100 --eval-n 100 --save-every 100

In [ ]:
# Score the untuned base model and the adapter on the same bench sample (860 examples each).
!python {PKG}/ml/b5_eval.py run --data {DATA} --out /kaggle/working/eval_base.json
!python {PKG}/ml/b5_eval.py run --data {DATA} --adapter /kaggle/working/b5_run1/adapter_final --out /kaggle/working/eval_tuned.json

In [ ]:
import json
for name in ("base", "tuned"):
    r = json.load(open(f"/kaggle/working/eval_{name}.json"))
    print(name.upper())
    for t, v in r["accuracy"].items():
        print(f"  {t:7s} acc {v['acc']:.3f}  (n={v['n']}, chance {v['chance']}, always-commonest {v['always_answer_the_commonest']:.3f})")
    print("  unreadable replies:", r["unreadable_replies"], "| caption names country/season:", r["caption_mentions_rate"])

In [ ]:
# Bundle the adapters, logs and both result files: download this zip from the Output tab.
import shutil
for f in ("eval_base.json", "eval_tuned.json"):
    shutil.copy(f"/kaggle/working/{f}", f"/kaggle/working/b5_run1/{f}")
shutil.make_archive("/kaggle/working/b5_run1_bundle", "zip", "/kaggle/working", "b5_run1")
print(round(os.path.getsize("/kaggle/working/b5_run1_bundle.zip") / 2**20, 1), "MiB")